# Variable Scope — The LEGB Rule

**Scope** determines where a variable is visible (accessible) in your code.

Python uses the **LEGB** rule to look up variable names:

| Letter | Scope | Description |
|--------|-------|-------------|
| **L** | Local | Inside the current function |
| **E** | Enclosing | In the enclosing function (for nested functions) |
| **G** | Global | At the top level of the module/file |
| **B** | Built-in | Python's built-in names (like `print`, `len`) |

Python searches in this order: L → E → G → B. It uses the **first match** it finds.

## 1. Local Scope

Variables defined **inside** a function are **local** — they only exist while the function runs.

In [ ]:
def my_function():
    x = 10  # local variable
    print(f'Inside function: x = {x}')

my_function()

# x doesn't exist outside the function
try:
    print(x)
except NameError as e:
    print(f'Error: {e}')

## 2. Global Scope

Variables defined at the **module level** (outside any function) are global — visible everywhere.

In [ ]:
# Global variable
greeting = 'Hello'

def say_hi(name):
    # Can READ the global variable
    print(f'{greeting}, {name}!')

say_hi('Alice')
say_hi('Bob')

In [ ]:
# Local variable shadows the global
x = 'global'

def show_x():
    x = 'local'  # creates a NEW local variable; doesn't touch the global
    print(f'Inside: {x}')

show_x()
print(f'Outside: {x}')  # still 'global'

## 3. The `global` Keyword

To **modify** a global variable from inside a function, you must declare it with `global`.

In [ ]:
counter = 0

def increment():
    global counter   # tell Python: 'counter' refers to the global one
    counter += 1
    print(f'Counter: {counter}')

increment()
increment()
increment()
print(f'Final counter: {counter}')

In [ ]:
# Without global — causes an UnboundLocalError
total = 0

def add_without_global(n):
    try:
        total += n  # Python sees 'total' on the left → assumes local → but it's not defined yet!
    except UnboundLocalError as e:
        print(f'Error: {e}')

add_without_global(5)

## 4. Enclosing Scope (Nested Functions)

When you define a function **inside** another function, the inner function has access to the **outer** (enclosing) function's variables.

In [ ]:
def outer():
    message = 'Hello from outer!'  # enclosing variable
    
    def inner():
        print(message)  # can read the enclosing variable
    
    inner()

outer()

In [ ]:
# LEGB in action — same name at different levels
x = 'global x'

def outer():
    x = 'enclosing x'
    
    def inner():
        x = 'local x'
        print(f'inner: {x}')   # L → found here
    
    inner()
    print(f'outer: {x}')       # E → found here

outer()
print(f'module: {x}')          # G → found here

## 5. The `nonlocal` Keyword

`nonlocal` lets an inner function **modify** a variable in the enclosing (outer) function's scope.

In [ ]:
def make_counter():
    count = 0  # enclosing variable
    
    def increment():
        nonlocal count   # refer to 'count' in make_counter's scope
        count += 1
        return count
    
    return increment

counter = make_counter()
print(counter())  # 1
print(counter())  # 2
print(counter())  # 3

# Each counter has its own 'count'
counter2 = make_counter()
print(counter2()) # 1 — fresh counter

## 6. Built-in Scope

Python has many built-in names: `print`, `len`, `range`, `type`, `int`, `list`, etc.  
These are always available — they're at the outermost scope (B in LEGB).

In [ ]:
# Built-in names are always accessible
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
print(len(numbers))   # built-in len
print(max(numbers))   # built-in max
print(sum(numbers))   # built-in sum

# You can see all builtins
import builtins
builtin_names = [name for name in dir(builtins) if not name.startswith('_')]
print(f'\nNumber of built-in names: {len(builtin_names)}')
print('First 10:', builtin_names[:10])

## 7. Scope Lookup Order Visualized

```
Built-in (B): print, len, range, int, ...
    ^
Global (G):   x = 'global'
    ^
Enclosing (E): x = 'enclosing'  ← only in nested functions
    ^
Local (L):    x = 'local'       ← checked first
```

Python starts at L and works up. The first match wins.

In [ ]:
# Demonstrating the full lookup order
name = 'global'       # G

def level1():
    name = 'enclosing'  # E
    
    def level2():
        # name = 'local'  # uncomment to use L
        print(f'level2 sees: {name}')  # finds E
    
    level2()
    print(f'level1 sees: {name}')  # finds L (its own)

level1()
print(f'module sees: {name}')  # finds G

## Common Mistakes

### Mistake 1: Expecting Local Changes to Affect the Global

In [ ]:
score = 100

# BAD — thinks it's modifying the global, but creates a local copy
def reset_score_bad():
    score = 0  # creates a new LOCAL variable!

reset_score_bad()
print(f'Score after reset (bad): {score}')  # still 100!

# GOOD — use global keyword
def reset_score_good():
    global score
    score = 0

reset_score_good()
print(f'Score after reset (good): {score}')  # 0

### Mistake 2: Accidentally Shadowing a Built-in

In [ ]:
# BAD — 'list' and 'len' are built-in names, don't overwrite them
# list = [1, 2, 3]  # now list() constructor is broken!
# len = 5           # now len() function is broken!

# GOOD — use descriptive names
my_list = [1, 2, 3]
my_len = 5
print(len(my_list))  # built-in len still works

### Mistake 3: Referencing a Variable Before Assignment in a Function

In [ ]:
x = 10

def broken():
    try:
        print(x)  # Python already decided 'x' is local (because of the next line)
        x = 20    # this makes Python treat all 'x' here as local
    except UnboundLocalError as e:
        print(f'UnboundLocalError: {e}')

broken()

# Fix: either don't assign x in the function, or use 'global x'
def fixed():
    global x
    print(x)  # reads global x = 10
    x = 20    # modifies global x

fixed()
print(f'x is now: {x}')

## Exercises

### Exercise 1 (Easy): Trace the Scope
What will the following code print? Predict first, then run it.

In [ ]:
# Predict the output before running!
x = 1

def func_a():
    x = 2
    print('func_a:', x)

def func_b():
    print('func_b:', x)

func_a()
func_b()
print('global:', x)

### Exercise 2 (Medium): Fix the Bug
The function below is supposed to keep a running total. Fix it.

In [ ]:
running_total = 0

def add_to_total(amount):
    # BUG: this doesn't actually update the global running_total
    running_total = running_total + amount

# Your fix here


In [ ]:
# Solution
running_total = 0

def add_to_total(amount):
    global running_total
    running_total += amount

add_to_total(10)
add_to_total(25)
add_to_total(5)
print(f'Running total: {running_total}')  # 40

### Exercise 3 (Hard): Closure-Based Accumulator
Write a function `make_accumulator(start)` that returns a function. Each time the returned function is called with a number, it adds it to a running total (starting at `start`) and returns the new total.

In [ ]:
# Your solution here


In [ ]:
# Solution
def make_accumulator(start=0):
    total = start
    
    def add(amount):
        nonlocal total
        total += amount
        return total
    
    return add

acc = make_accumulator(100)
print(acc(10))   # 110
print(acc(25))   # 135
print(acc(-50))  # 85

# Independent accumulator
acc2 = make_accumulator(0)
print(acc2(5))   # 5

## Mini-Project: Counter Using Closure

Build a counter factory that creates independent counters, each with their own state.

In [ ]:
def make_counter(name, step=1):
    """
    Create a named counter.
    Returns a dict of functions: increment, decrement, reset, value.
    """
    count = 0
    
    def increment():
        nonlocal count
        count += step
        return count
    
    def decrement():
        nonlocal count
        count -= step
        return count
    
    def reset():
        nonlocal count
        count = 0
        return count
    
    def value():
        return count
    
    def show():
        print(f'[{name}] count = {count}')
    
    return {'increment': increment, 'decrement': decrement,
            'reset': reset, 'value': value, 'show': show}


# Create two independent counters
c1 = make_counter('visits', step=1)
c2 = make_counter('score', step=10)

c1['increment']()
c1['increment']()
c1['increment']()
c1['show']()   # [visits] count = 3

c2['increment']()
c2['increment']()
c2['show']()   # [score] count = 20

c1['decrement']()
c1['show']()   # [visits] count = 2

c1['reset']()
c1['show']()   # [visits] count = 0
c2['show']()   # [score] count = 20 (unaffected)